In [ ]:
! pip show langchain

Name: langchain
Version: 1.0.1
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/ahmed/miniconda3/envs/ai/lib/python3.10/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [39]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent

In [2]:
load_dotenv()

True

In [40]:


# 1. Configuration (Phase 2)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7,api_key=os.getenv("OPENAI_API_KEY"))
trvily_api_key = os.getenv("TAVILY_API_KEY")

# 2. Tools - The "Travel Planner" logic (Phase 4)
# This allows the AI to search for real flights/hotels
search_tool = TavilySearchResults(k=3, api_key=trvily_api_key) 
tools = [search_tool]

# 3. Itinerary Chain Logic (Phase 3)
# We define a system prompt that guides the AI's behavior
system_prompt = """
You are an expert Travel Planner. 
Your goal is to create a structured, realistic itinerary based on user input.
1. Use the search tool to find current events or hotel recommendations.
2. Provide a day-by-day breakdown.
3. Include estimated costs and travel tips.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 4. Construct the Agent
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5. Execution
def generate_travel_plan(destination, duration, budget):
    user_query = f"Plan a {duration} trip to {destination} with a {budget} budget."
    response = agent_executor.invoke({"input": user_query})
    return response["output"]



In [6]:
# Example Usage:
if __name__ == "__main__":
    plan = generate_travel_plan("Bangladesh", "5 days", "CoxsBazar")
    print("\n--- YOUR TRAVEL PLAN ---")
    print(plan)



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': "Cox's Bazar hotels and current events"}`


[{'title': "THE 10 BEST Hotels in Cox's Bazar, Bangladesh 2026 (from $18)", 'url': 'https://www.tripadvisor.com/Hotels-g667467-Cox_s_Bazar_Chittagong_Division-Hotels.html', 'content': "View deals from $18 per night, see photos and read reviews for the best Cox's Bazar hotels from travelers like you - then compare", 'score': 0.9997508}, {'title': 'Latest News, Photos, Videos on Cox Bazar - NDTV.COM', 'url': 'https://www.ndtv.com/topic/cox-bazar', 'content': '"Sudden Attack" On Bangladesh Airbase, Locals Clash With Airmen, 1 Killed ... A man was killed and several others injured in a clash Monday between a group of', 'score': 0.99937373}, {'title': 'https://coxsbazarlife.com/pm-inquires-about-bangladeshi-expats ...', 'url': 'https://www.instagram.com/p/DVWT3tBCWpN/', 'content': "One of the key organizations in Cox's Bazar's tourism industry, Leisure 

### add tools

In [8]:
from langchain_core.tools import tool

@tool
def calculate_budget(daily_hotel: float, daily_food: float, flights: float, days: int) -> str:
    """Calculates the total trip cost based on daily estimates."""
    total = (daily_hotel + daily_food) * days + flights
    return f"The total estimated budget for {days} days is ${total:,.2f}."

# Add this to your tools list:
tools = [search_tool, calculate_budget]

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain.agents import create_agent

# 2. Define the Itinerary Schema (Phase 3 Logic)
class DayPlan(BaseModel):
    day: int = Field(description="The day number")
    activity: str = Field(description="Main activity for the day")
    estimated_cost: float = Field(description="Cost for this day in USD")

class TravelPlan(BaseModel):
    destination: str
    total_budget: float
    itinerary: List[DayPlan]


In [21]:
travel_agent = create_agent(
    model=llm,
    tools=tools,
    response_format=TravelPlan, # Forces structured JSON output
    system_prompt=(
"""
You are an expert Travel Planner. 
Your goal is to create a structured, realistic itinerary based on user input.
1. Use the search tool to find current events or hotel recommendations.
2. Provide a day-by-day breakdown.
3. Include estimated costs and travel tips.
"""
    )
)

In [22]:
# 4. Invoke the Agent
response = travel_agent.invoke({"messages": [("user", "Plan a 5-day trip to Bangladesh in Cox's Bazar with a $300 budget.")]})

# Accessing the structured data
plan = response["structured_response"]
print(f"Destination: {plan.destination}")
for day in plan.itinerary:
    print(f"Day {day.day}: {day.activity} (${day.estimated_cost})")

Destination: Cox's Bazar, Bangladesh
Day 1: Arrival in Cox's Bazar, settle in at the hotel, and explore the local area. ($20.0)
Day 2: Visit Cox's Bazar beach for sunbathing and swimming, enjoy local street food. ($25.0)
Day 3: Day trip to Himchari National Park for hiking and nature exploration. ($30.0)
Day 4: Explore local markets and shops, try seafood restaurants along the beach. ($25.0)
Day 5: Relax at the beach, revisit favorite spots before departure. ($25.0)


### more tools

In [41]:
load_dotenv(override=True)

True

In [42]:
from langchain_core.tools import tool
from langchain_community.utilities import OpenWeatherMapAPIWrapper

@tool
def check_weather_safety(destination: str) -> str:
    """Checks current weather. Returns 'Safe' or 'Warning' with details."""
    weather = OpenWeatherMapAPIWrapper(openweathermap_api_key=os.getenv("OPENWEATHERMAP_API_KEY"))
    data = weather.run(destination)
    # Logic: If 'rain' in data, suggest indoor activities
    return f"Weather in {destination}: {data}. Advice: Proceed with the trip but carry an umbrella."


@tool
def calculate_final_costs(transport: float, hotel_daily: float, days: int, food_daily: float) -> dict:
    """Calculates total trip cost. Essential for Point 7."""
    total_stay = (hotel_daily + food_daily) * days
    grand_total = total_stay + transport
    return {
        "total_stay_cost": total_stay,
        "grand_total": grand_total,
        "currency": "BDT"
    }


In [34]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper


tavily_key = os.getenv("TAVILY_API_KEY")
search_tool = TavilySearchResults(k=3, api_key=tavily_key)

# Option B: Manual Wrapper (If Option A fails)
# wrapper = TavilySearchAPIWrapper(tavily_api_key=os.getenv("TAVILY_API_KEY"))
# search_tool = TavilySearchResults(api_wrapper=wrapper, k=3)

tools = [search_tool, check_weather_safety]

In [43]:
from pydantic import BaseModel, Field
from typing import List

class DayItinerary(BaseModel):
    day_number: int
    spots: List[str] = Field(description="Top 5 spots to visit for this day")
    local_food_suggestions: List[str] = Field(description="Point 4: Specific local food to try")
    activity_timing: str = Field(description="Point 6: Breakdown of time spent in area")
    estimated_daily_cost: float

class FinalTravelPlan(BaseModel):
    destination: str
    weather_status: str
    daily_plans: List[DayItinerary]
    total_estimated_budget: float = Field(description="Point 7: Final solid total cost")

In [44]:
# Create the agent using the 2026 factory function
from langchain.agents import create_agent

travel_agent = create_agent(
    model=llm,
    tools=tools,
    response_format=FinalTravelPlan, # This enforces your 7 points!
    system_prompt=(
    "You are a realistic Travel Planner. "
    "IMPORTANT: Daily costs must be realistic and vary day-by-day. "
    "Do not simply divide the total budget by the number of days. "
    "For example, a day with a Safari Park visit should cost more than a day at the beach."

    )
)

In [45]:
! pip install pyowm

In [46]:
response = travel_agent.invoke({"messages": [("user", "Plan a 5-day trip to Bangladesh in Cox's Bazar with a $300 budget.")]})


In [47]:
response

{'messages': [HumanMessage(content="Plan a 5-day trip to Bangladesh in Cox's Bazar with a $300 budget.", additional_kwargs={}, response_metadata={}, id='d693d7fb-d36e-45e0-b92e-c4fe014c8602'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 259, 'total_tokens': 283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6bedc898d2', 'id': 'chatcmpl-DGfNvdwbwHmQL4xuy8L2luzUNxvMv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cc6f8-8a65-72a2-b09c-52e4817225f3-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': "Cox's Bazar travel guide"}, 'id': 'call_kaftjdD0NNGjshyAVDVRpMJt', 'type': 'tool_c

In [48]:
response["structured_response"]

FinalTravelPlan(destination="Cox's Bazar, Bangladesh", weather_status='Warm and sunny with mild evenings during the trip', daily_plans=[DayItinerary(day_number=1, spots=['Laboni Beach', 'Sugandha Beach', 'Marine Drive', 'Himchari National Park', 'Local Marketplace'], local_food_suggestions=['Fish Curry', 'Prawn Pulao', 'Coconut Water', 'Street Snacks'], activity_timing='Morning to Evening', estimated_daily_cost=50.0), DayItinerary(day_number=2, spots=['Inani Beach', 'Radiant Fish World', 'Darianagar Eco Park', 'Burmese Market', 'Sunset at Himchari'], local_food_suggestions=['Grilled Fish', 'Seafood Pasta', 'Fried Calamari', 'Fresh Fruit Juices'], activity_timing='Morning to Evening', estimated_daily_cost=70.0), DayItinerary(day_number=3, spots=['Himchari Waterfall', '100 Feet Buddha Temple', 'Kutubdia Lighthouse', 'Beach Horse Riding', 'Parasailing'], local_food_suggestions=['Fish Tikka', 'Seafood BBQ', 'Sandwiches', 'Tropical Smoothies'], activity_timing='Morning to Evening', estimate

In [49]:
# Accessing the structured data (Point 7)
plan = response["structured_response"]

print(f"--- {plan.destination} Plan ---")
print(f"Weather Status: {plan.weather_status}")
for day in plan.daily_plans:
    print(f"\nDay {day.day_number}:")
    print(f"Spots: {', '.join(day.spots)}")
    print(f"Food: {', '.join(day.local_food_suggestions)}")
    print(f"Schedule: {day.activity_timing}")
    print(f"Daily-Cost: {day.estimated_daily_cost}")

print(f"\nTOTAL SOLID COST: ${plan.total_estimated_budget}")

--- Cox's Bazar, Bangladesh Plan ---
Weather Status: Warm and sunny with mild evenings during the trip

Day 1:
Spots: Laboni Beach, Sugandha Beach, Marine Drive, Himchari National Park, Local Marketplace
Food: Fish Curry, Prawn Pulao, Coconut Water, Street Snacks
Schedule: Morning to Evening
Daily-Cost: 50.0

Day 2:
Spots: Inani Beach, Radiant Fish World, Darianagar Eco Park, Burmese Market, Sunset at Himchari
Food: Grilled Fish, Seafood Pasta, Fried Calamari, Fresh Fruit Juices
Schedule: Morning to Evening
Daily-Cost: 70.0

Day 3:
Spots: Himchari Waterfall, 100 Feet Buddha Temple, Kutubdia Lighthouse, Beach Horse Riding, Parasailing
Food: Fish Tikka, Seafood BBQ, Sandwiches, Tropical Smoothies
Schedule: Morning to Evening
Daily-Cost: 80.0

Day 4:
Spots: Teknaf, Naf River Boat Ride, Local Fishing Village, Buddhist Monasteries in Ramu, Evening Beach Stroll
Food: Shrimp Biryani, Vegetable Samosas, Local Fish Sandwich, Chilled Coconut Drink
Schedule: Morning to Evening
Daily-Cost: 60.0

D